# 🔎 Binary Search on Answer Space

Standard binary search finds a specific number in a sorted array. **Binary Search on Answer Space** is a more advanced pattern that HackerRank frequently uses for optimization problems. Instead of searching an array, you search the range of possible answers.

Here is a classic scenario adapted to a database engineering context.

## 🗄️ Minimum Server Capacity

You are migrating a massive database. The data is divided into an array of chunks, where each element represents the size of a chunk in gigabytes (GB).

The migration must be completed within exactly `D` days, and the chunks must be processed in the order they appear. Each day, the server processes chunks sequentially until adding the next chunk would exceed its daily maximum capacity, `C`.

**Goal:** Find the minimum daily server capacity `C` required to finish the migration within `D` days.

In [2]:
chunks = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
days = 5

> **Expected output:** `15`
>
> With a capacity of 15, the migration is processed as follows:
> - **Day 1:** `[1, 2, 3, 4, 5]`
> - **Day 2:** `[6, 7]`
> - **Day 3:** `[8]`
> - **Day 4:** `[9]`
> - **Day 5:** `[10]`

### ⚠️ The Trap: Brute Force

- **Lowest possible capacity:** `10`, the size of the largest individual chunk. With a capacity of 9, the 10 GB chunk could never be processed.
- **Highest possible capacity:** `55`, the sum of all chunks. At this capacity, all chunks could be processed in one day.

A brute-force approach would test capacities `10`, `11`, `12`, and so on, calculating the number of days required for each one until it finds a capacity that takes `≤ 5` days. This results in a **Time Limit Exceeded (TLE)** error on large datasets.

### ✅ Binary Search the Answer

Because the range of possible answers `[10, 55]` is sorted, binary search can find the optimal capacity efficiently.

1. **Set the boundaries:** `left = max(chunks)` and `right = sum(chunks)`.
2. **Find the midpoint:** Test the middle capacity.
3. **Verify the capacity:** Use a helper function to simulate the migration with that capacity.
4. **If it finishes in `≤ D` days:** The midpoint is a valid answer. Since we want the minimum, search the lower half by setting `right = mid - 1` to see whether a smaller capacity works.
5. **If it takes `> D` days:** The capacity is too small. Search the upper half by setting `left = mid + 1`.

In [3]:
def min_server_capacity(chunks, days):
    left, right = max(chunks), sum(chunks)
    def can_finish(capacity):
        current_load = 0
        required_days = 1

        for chunk in chunks:
            if current_load + chunk > capacity:
                required_days += 1
                current_load = chunk
            else:
                current_load += chunk

        return required_days <= days

    optimal_capacity = right
    while left < right:
        mid = (left + right) // 2
        if can_finish(mid):
            optimal_capacity = mid
            right = mid
        else:
            left = mid + 1
    return optimal_capacity

In [4]:
chunks = [5,4,8,9,15,30,21]
days = 3 # we need to finish the chunks in 3 days

result = min_server_capacity(chunks, days)
print(f"The minimum server capacity required to finish the chunks in {days} days is: {result}")

The minimum server capacity required to finish the chunks in 3 days is: 41


💡 **Practical application:** We can use this approach to estimate costs based on file charges and the number of days needed to process the chunks.

### ⏱️ Complexity Analysis

- **Time complexity:** `O(N log M)`, where `N` is the number of chunks and `M` is the size of the search space, `sum(chunks) - max(chunks)`.
  - Binary search takes `O(log M)` steps.
  - At each step, we iterate through the chunks once in `O(N)` time to verify the capacity.
  - This is efficient for the large datasets used in HackerRank's hidden tests.
- **Space complexity:** `O(1)` auxiliary space.

# 🧠 Kadane’s Algorithm (Dynamic Programming)

Dynamic Programming (DP) can seem intimidating, but the core idea is to break a problem into smaller subproblems, solve them, and remember the results so they do not need to be recalculated.

One of the most common entry-level DP problems is **Maximum Subarray Sum**, which can be solved using Kadane’s Algorithm.

## 🗄️ Maximum Server Load Spike

You are analyzing a log of minute-by-minute net changes in a database server’s active connection count. The array contains both positive numbers (connection surges) and negative numbers (dropped connections).

**Goal:** Find the contiguous timeframe (subarray) that produces the maximum net surge in connections, and return that maximum sum.

In [5]:
arr = [-2, 1, -3, 4, -1, 2, 1, -5, 4]

> **Expected output:** `6`
>
> The contiguous subarray `[4, -1, 2, 1]` produces the maximum sum of `6`.

### ⚠️ The Trap: Brute Force

The beginner approach uses nested loops to calculate the sum of every possible contiguous subarray.

- **Time complexity:** `O(N²)` or even `O(N³)`.
- **Why it fails:** HackerRank can return a **Time Limit Exceeded (TLE)** error for arrays larger than 10,000 elements.

### ✅ Kadane’s Algorithm in `O(N)`

Kadane’s Algorithm optimizes the solution to a single pass. As you iterate through the array, you maintain a running sum and make a dynamic programming decision at every number:

> Is it better to add the current number to the existing sequence, or is the current sequence so negative that it is better to start a new sequence here?

- **`max_current`:** The best sum ending at the current position. Calculate it as `max(current_number, max_current + current_number)`.
- **`max_global`:** The highest value of `max_current` seen across the entire array.

In [6]:
def max_subarray_sum(arr):
    max_current = arr[0]
    max_global = arr[0]

    for i in range(1, len(arr)):
        print(f"max_current before index {i}: {max_current} (considering element {arr[i]} or starting new subarray {max_current + arr[i]})")
        max_current = max(arr[i], max_current + arr[i])
        print(f"Current max at index {i}: {max_current}")

        if max_current > max_global:
            max_global = max_current

    return max_global

In [7]:
print(f"The maximum subarray sum is: {max_subarray_sum(arr)}")

max_current before index 1: -2 (considering element 1 or starting new subarray -1)
Current max at index 1: 1
max_current before index 2: 1 (considering element -3 or starting new subarray -2)
Current max at index 2: -2
max_current before index 3: -2 (considering element 4 or starting new subarray 2)
Current max at index 3: 4
max_current before index 4: 4 (considering element -1 or starting new subarray 3)
Current max at index 4: 3
max_current before index 5: 3 (considering element 2 or starting new subarray 5)
Current max at index 5: 5
max_current before index 6: 5 (considering element 1 or starting new subarray 6)
Current max at index 6: 6
max_current before index 7: 6 (considering element -5 or starting new subarray 1)
Current max at index 7: 1
max_current before index 8: 1 (considering element 4 or starting new subarray 5)
Current max at index 8: 5
The maximum subarray sum is: 6


### ⏱️ Complexity Analysis

- **Time complexity:** `O(N)`. We iterate through the log exactly once.
- **Space complexity:** `O(1)`. We use only two variables, regardless of whether the log contains hundreds or millions of records. This is ideal for database stream processing.

# 🎯 Activity Selection (Greedy Algorithms)

Greedy algorithms build a solution piece by piece, always choosing the next option that offers the most immediate benefit. In backend and database engineering, a frequently tested greedy pattern is **Interval Scheduling**, often framed as scheduling CPU threads, database backup jobs, or meeting rooms.

## 🗄️ Database Backup Scheduling

You are given a list of scheduled database maintenance jobs. Each job is represented by two integers: `[start_time, end_time]`.

Your server can execute only one job at a time.

**Goal:** Find the maximum number of jobs the server can complete without overlapping. If one job ends at time `3`, another job can start at exactly time `3`.

In [8]:
jobs = [[1, 4], [2, 3], [4, 6], [3, 5], [5, 7]]

> **Expected output:** `3`
>
> The optimal schedule is to run `[2, 3]`, then `[3, 5]`, then `[5, 7]`.

### ⚠️ The Trap: Choosing the Wrong Greedy Rule

The intuitive approach is often to sort jobs by their `start_time` or by the **shortest duration**. Both choices can fail on edge cases.

- **Choosing the earliest start time:** A job running from `[1, 100]` could block out every other job.
- **Choosing the shortest duration:** A job running from `[4, 5]` might overlap with two useful jobs such as `[2, 4]` and `[5, 7]`.

### ✅ The Correct Greedy Choice: Earliest Finish Time

Always choose the job that ends earliest. Finishing a job as soon as possible leaves the maximum amount of free time for subsequent jobs.

1. Sort the jobs by `end_time` only.
2. Track `last_end_time`, the end time of the most recently scheduled job.
3. Iterate through the sorted jobs. If the next job’s `start_time` is `≥ last_end_time`, schedule it, update `last_end_time`, and increase the count.

In [9]:
def max_non_overlapping_jobs(jobs):
    if not jobs:
        return 0

    jobs.sort(key=lambda x: x[1])

    count = 0
    last_end_time = float('-inf')

    for start,end in jobs:
        print(f"Considering job with start: {start}, end: {end}. Last end time: {last_end_time}")
        if start >= last_end_time:
            print(f"Job can be scheduled. Incrementing count and updating last end time to {end}.\n")
            count += 1
            last_end_time = end
        else:
            print(f"Job overlaps with the last scheduled job. Skipping this job.\n")

    return count

In [10]:
max = max_non_overlapping_jobs(jobs)
print(f"Maximum number of non-overlapping jobs: {max}")

Considering job with start: 2, end: 3. Last end time: -inf
Job can be scheduled. Incrementing count and updating last end time to 3.

Considering job with start: 1, end: 4. Last end time: 3
Job overlaps with the last scheduled job. Skipping this job.

Considering job with start: 3, end: 5. Last end time: 3
Job can be scheduled. Incrementing count and updating last end time to 5.

Considering job with start: 4, end: 6. Last end time: 5
Job overlaps with the last scheduled job. Skipping this job.

Considering job with start: 5, end: 7. Last end time: 5
Job can be scheduled. Incrementing count and updating last end time to 7.

Maximum number of non-overlapping jobs: 3


### ⏱️ Complexity Analysis

- **Time complexity:** `O(N log N)` for sorting. Iterating through the jobs takes `O(N)`.
- **Space complexity:** `O(1)` or `O(N)`, depending on the language’s sorting implementation. Python’s Timsort uses `O(N)` extra space in the worst case.

# 🪟 The Sliding Window (Variable Size)

The Sliding Window technique tracks a subset of data within a larger array or string. In database engineering, similar logic is used to analyze rolling time-series data—for example, finding the maximum traffic spike in any five-minute rolling window.

On HackerRank, one of the most common variations is the **variable-size sliding window**.

## 🔎 Longest Unique Query Sequence

You are given a string of transaction-type codes (characters) submitted to a database.

**Goal:** Find the length of the longest contiguous substring that contains no repeating characters.

In [2]:
transactions = "abcabcbb"

> **Expected output:** `3`
>
> The longest sequences without duplicates are `abc`, `bca`, and `cab`, all of which have a length of `3`.

### ⚠️ The Trap: Brute Force

The beginner approach generates every possible substring and checks each one for duplicates.

- **Time complexity:** `O(N³)` (or `O(N²)` if heavily optimized).
- **Why it fails:** HackerRank may time out on strings containing 100,000 or more characters.

### ✅ Sliding Window + Hash Map

Instead of restarting for every substring, use two pointers—`left` and `right`—to create a window of valid characters. Expand the window by moving `right` one step at a time.

If `right` encounters a character already seen inside the current window, the window becomes invalid. Shrink it immediately by moving `left` to just after that character’s previous occurrence.

- **Dictionary (`seen`):** Stores the last known index for each character, in the form `{char: index}`.
- **`max_len`:** Tracks the largest valid window found.
- **`left` pointer:** Marks the start of the current valid window.

In [5]:
def length_of_longest_substring(s):
    seen = {}
    left = 0
    max_length = 0


    for right,char in enumerate(s):
        print(f"seen[char]: {seen.get(char, 'not seen yet')}")
        print(f"the right pointer is at index {right} with character '{char}' and the left pointer is at index {left}.")
        if char in seen and seen[char] >= left:
            print(f"Character '{char}' has been seen before at index {seen[char]}. Moving left pointer to {seen[char] + 1}.\n")
            left = seen[char] + 1
        else:
            print("\n")

        seen[char] = right
        current_window_len = right - left + 1
        max_length = max(max_length, current_window_len)
        print(f"Current window length: {current_window_len}. Max length so far: {max_length}.\n")
        



In [6]:
length_of_longest_substring(transactions)

seen[char]: not seen yet
the right pointer is at index 0 with character 'a' and the left pointer is at index 0.


Current window length: 1. Max length so far: 1.

seen[char]: not seen yet
the right pointer is at index 1 with character 'b' and the left pointer is at index 0.


Current window length: 2. Max length so far: 2.

seen[char]: not seen yet
the right pointer is at index 2 with character 'c' and the left pointer is at index 0.


Current window length: 3. Max length so far: 3.

seen[char]: 0
the right pointer is at index 3 with character 'a' and the left pointer is at index 0.
Character 'a' has been seen before at index 0. Moving left pointer to 1.

Current window length: 3. Max length so far: 3.

seen[char]: 1
the right pointer is at index 4 with character 'b' and the left pointer is at index 1.
Character 'b' has been seen before at index 1. Moving left pointer to 2.

Current window length: 3. Max length so far: 3.

seen[char]: 2
the right pointer is at index 5 with character 'c

### ⏱️ Complexity Analysis

- **Time complexity:** `O(N)`. Both pointers (`left` and `right`) move only forward, so a string of a million characters is processed in one pass.
- **Space complexity:** `O(M)`, where `M` is the number of unique characters in the alphabet. For example, `O(26)` for lowercase English letters, which is effectively `O(1)` constant space.

# 📊 Prefix Sums (Range Queries)

Alongside the other major algorithmic paradigms, HackerRank frequently uses **Prefix Sums** in database and data engineering assessments. This pattern tests your ability to optimize multiple consecutive queries against a static dataset—essentially building a simple materialized view.

## 🗄️ High-Volume Range Queries

You are given an array representing the number of deadlocks a database experienced each day over a year, along with a large list of `Q` queries. Each query is a pair of indices `[L, R]` and asks for the total number of deadlocks between day `L` and day `R`, inclusive.

In [7]:
deadlocks = [3, 1, 4, 1, 5, 9, 2, 6]
queries = [[0, 2], [2, 5], [4, 7]]

> **Expected output:** `[8, 19, 22]`
>
> For example, the sum for indices `0` to `2` is `3 + 1 + 4 = 8`.

### ⚠️ The Trap: Brute Force

For every query `[L, R]`, a loop sums the elements from `L` to `R`.

- **Time complexity:** `O(Q × N)`.
- **Why it fails:** With 100,000 days and 100,000 queries, this can require 10 billion operations. HackerRank will likely time out on a large test case.

### ✅ Precompute the Prefix Sum

Instead of recalculating each range sum, iterate through the array once to build `prefix_array`, where each index stores the cumulative sum of all elements up to that point.

Once it is built, any range sum `[L, R]` can be found in `O(1)` time using subtraction:

\[
\text{Sum}(L, R) = \text{prefix\_array}[R] - \text{prefix\_array}[L-1]
\]

In [ ]:
def process_range_queries(arr, queries):
    results = []
    if not arr:
        return []
    for q in queries:
        L, R = q
        sum_elements = 0
        for i in range(L, R + 1):
            sum_elements = sum_elements + arr[i]
        results.append(sum_elements)
    return results


In [9]:
process_range_queries(deadlocks, queries)

[8, 19, 22]

### ⚠️ Brute-Force Complexity

- **Time complexity:** `O(Q × N)` in the worst case, because each query could touch every element.
- **Work estimate:** `Q` queries × up to `N` elements per query.
- **For `N, Q = 10⁵`:** Up to `10¹⁰` operations, which leads to **Time Limit Exceeded (TLE)** on most judges.
- **Space complexity:** `O(1)` extra space, apart from the results list.

In [13]:
def process_range_queries(arr, queries):
    if not arr:
        return []

    prefix_sum = [0] * (len(arr))
    prefix_sum[0] = arr[0]
    for i in range(1, len(arr)):
        print(f"Calculating prefix sum at index {i}: prefix_sum[{i}] = prefix_sum[{i - 1}] + arr[{i}] = {prefix_sum[i - 1]} + {arr[i]} = {prefix_sum[i - 1] + arr[i]}")
        prefix_sum[i] = prefix_sum[i - 1] + arr[i]

    results = []
    for q in queries:
        L, R = q
        if L == 0:
            print(f"Query from {L} to {R}: Since L is 0, the sum is simply prefix_sum[{R}] = {prefix_sum[R]}")
            results.append(prefix_sum[R])
        else:
            print(f"Query from {L} to {R}: The sum is prefix_sum[{R}] - prefix_sum[{L - 1}] = {prefix_sum[R]} - {prefix_sum[L - 1]} = {prefix_sum[R] - prefix_sum[L - 1]}")
            results.append(prefix_sum[R] - prefix_sum[L - 1])

    return results

In [14]:
process_range_queries(deadlocks, queries)

Calculating prefix sum at index 1: prefix_sum[1] = prefix_sum[0] + arr[1] = 3 + 1 = 4
Calculating prefix sum at index 2: prefix_sum[2] = prefix_sum[1] + arr[2] = 4 + 4 = 8
Calculating prefix sum at index 3: prefix_sum[3] = prefix_sum[2] + arr[3] = 8 + 1 = 9
Calculating prefix sum at index 4: prefix_sum[4] = prefix_sum[3] + arr[4] = 9 + 5 = 14
Calculating prefix sum at index 5: prefix_sum[5] = prefix_sum[4] + arr[5] = 14 + 9 = 23
Calculating prefix sum at index 6: prefix_sum[6] = prefix_sum[5] + arr[6] = 23 + 2 = 25
Calculating prefix sum at index 7: prefix_sum[7] = prefix_sum[6] + arr[7] = 25 + 6 = 31
Query from 0 to 2: Since L is 0, the sum is simply prefix_sum[2] = 8
Query from 2 to 5: The sum is prefix_sum[5] - prefix_sum[1] = 23 - 4 = 19
Query from 4 to 7: The sum is prefix_sum[7] - prefix_sum[3] = 31 - 9 = 22


[8, 19, 22]

### ⏱️ Complexity Analysis

- **Time complexity:** `O(N + Q)`. Iterate through the array of length `N` once, then process each of the `Q` queries in constant time.
- **Space complexity:** `O(N)` to store the `prefix_sum` array.

## 🧭 Other Niche Algorithm Patterns to Recognize

### Topological Sort (Kahn’s Algorithm)

If a problem asks for the **valid execution order** of tasks with dependencies—for example, Task A must finish before Task B—build a **Directed Acyclic Graph (DAG)** and use topological sort.

### Bitwise XOR for Singles

If every ID in a database appears exactly twice except one ID that appears once, and you must find it in `O(1)` space, use `result ^= num` while iterating through the array. XORing a number with itself cancels it to `0`, leaving only the unique number.

In [6]:
arr = [3, 1, 4, 1, 3]
result = 0
for x in arr:
    result ^= x
print(f"Result after XOR operation with 0: {result}")

Result after XOR operation with 0: 4
